# Lesson 20 · 🎨 A map of decompositions — breaking a complicated transformation into a few simple moves with LU / QR / SVD

> From *The Art of Linear Algebra*. Split a matrix into a product of a few pieces:

`A=LU` (elimination) · `A=QR` (orthogonalization) · `S=QΛQᵀ` (symmetric eigendecomposition) · `A=UΣVᵀ` (SVD).

**Learning goals**
1. Use `laviz`'s `show_factorization` to draw LU decomposition as the geometric steps "lower-triangular elimination × upper-triangular staircase"
2. Observe the orthogonality of Q in a QR decomposition (`QᵀQ = I`) and the upper-triangular structure of R, and understand what Gram-Schmidt orthogonalization means
3. Compare the geometric structure of the spectral theorem for symmetric matrices, `S = QΛQᵀ`, with the SVD of a general matrix, `A = UΣVᵀ`
4. Judge a matrix's effective rank from its sequence of singular values: keeping the first k singular values rebuilds the main information, and dropping the small ones filters out noise
5. Understand the low-rank SVD approximation of an Aurora mel spectrogram matrix: keep the large singular values, drop the small ones, and the main structure separates from the background noise

**Why it matters for Aurora**: the mel filter matrix in the MFCC extraction pipeline can be reduced with SVD, and QR decomposition gives numerically stable least squares; once you understand the geometry, you stop guessing when you tune the code.

← **Previous lesson** · [L19 · Matrix transformations illustrated](L19_visual_multiply.ipynb)

> Last lesson covered **matrix transformations illustrated**: rotation, scaling, shear and "a linear combination of columns", shown visually.
> This lesson explores **a map of decompositions**.

## This lesson's story: name → map of geometric moves

> **Where this lesson sits**: we match **the names LU / QR / SVD to their geometric moves**, **without full proofs**.
> Elimination details are in L15 and low-rank SVD is in L14 — here the pictures put all three "ways to split" on one map.

Matrix decomposition = breaking a complicated transformation into a few simple moves:
- **LU**: elimination written in matrix language (lower triangular × upper triangular)
- **QR**: first turn to orthogonal directions, then stretch
- **SVD**: rotate → stretch along the axes → rotate again (works for a matrix of any shape)

From the pictures, just remember "what each step does to the coordinate system"; when you need the detailed computation, go back to the corresponding lesson.

In [ ]:
# Aurora matplotlib bootstrap
from pathlib import Path
import sys

_root = None
_cwd = Path.cwd().resolve()
for _candidate in (_cwd, *_cwd.parents):
    if (_candidate / '_matplotlib_bootstrap.py').exists():
        _root = _candidate
        break
if _root is None:
    _notebooks_dir = _cwd / 'notebooks'
    if _notebooks_dir.exists():
        for _found in _notebooks_dir.rglob('_matplotlib_bootstrap.py'):
            _root = _found.parent
            break
if _root is not None and str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from _matplotlib_bootstrap import apply as _aurora_mpl_apply
_aurora_mpl_apply()


In [ ]:
import numpy as np
from laviz import style, show_factorization, heatmap
style(); print('Ready')

## Reading the notation: shape first, then the operation

Every factor in a decomposition is a matrix, and understanding the decomposition means understanding each factor's geometric effect in turn. In LU, L does row shears and U does upper-triangular scaling; in QR, Q is a rotation and R is an upper-triangular stretch; in SVD, U and V are rotations and Σ is a stretch along the axes.

## Hands-on: for linear-algebra objects, look at the shape first, then the meaning

Run the code below and pay attention to each factor's shape: why L and U are both square, why the columns of Q are pairwise orthogonal, and how the dimensions of U, Σ and Vᵀ in the SVD line up with the number of rows and columns of the original matrix A.

In [ ]:
import numpy as np

# QR decomposition: A = Q R (Q orthogonal, R upper triangular)
A = np.array([[1.,2.],[3.,4.],[5.,6.]])
Q, R = np.linalg.qr(A)
print('Q (orthonormal columns) =\n', np.round(Q, 4))
print('R (upper triangular) =\n', np.round(R, 4))
print('Q^T @ Q =\n', np.round(Q.T @ Q, 10))  # should be I
print('Reconstruction error =', np.max(np.abs(A - Q @ R)))


### Pause: what does "orthogonal" actually mean? Why is QᵀQ = I called orthogonal?

The "perpendicular" you learned in school means two lines meet at 90°, and the test is the dot product: two vectors $a, b$ are perpendicular $\iff a\cdot b = 0$. The Q above is a matrix, but its "orthogonality" is a direct generalization of that school fact — we're just checking many pairs of vectors at once.

Think of Q as "several columns standing side by side": each **column** of Q is a vector. The phrase "the columns of Q are pairwise orthogonal" actually unpacks into two things:

1. Take any two different columns $q_i, q_j$ of Q ($i\neq j$): they must be perpendicular, $q_i \cdot q_j = 0$.
2. Each column's own length is exactly 1 (a "unit vector"): $q_i \cdot q_i = 1$.

Now look at what the matrix product $Q^\top Q$ computes. Row $i$ of $Q^\top$ is column $i$ of $Q$ (transposing just lays the columns down as rows). Entry $(i,j)$ of a matrix product is "row $i$ of $Q^\top$" dotted with "column $j$ of $Q$" — that is, $q_i \cdot q_j$. In other words:

$$Q^\top Q \text{ is the whole table of "the dot product of every pair of columns", laid out at once.}$$

If all the columns are pairwise perpendicular (every off-diagonal entry is 0) and every column has length 1 (the diagonal is all 1s), this table is exactly the identity matrix $I$. Conversely, if you compute $Q^\top Q = I$, you can work backwards to "the columns are pairwise perpendicular with length 1" — that's why mathematics defines an "orthogonal matrix" by $Q^\top Q = I$: the formula is just the matrix way of writing "pairwise perpendicular + unit length".

Below we check this correspondence with two vectors worked by hand, and along the way explain why Q here is 3×2 (not square) — that comes from the **reduced QR** that numpy uses by default.

In [ ]:
import numpy as np

# 1) What does "orthogonal" actually mean? First, dot products of two concrete vectors by hand
q1 = np.array([0.6, 0.8])
q2 = np.array([-0.8, 0.6])
print('q1·q1 =', q1 @ q1, ' (squared length; =1 means q1 is a unit vector)')
print('q2·q2 =', q2 @ q2, ' (likewise, q2 is a unit vector too)')
print('q1·q2 =', q1 @ q2, ' (=0 means the two vectors are perpendicular)')

Qtoy = np.column_stack([q1, q2])
print('\nPut q1, q2 side by side as the two columns of Qtoy; QtoyᵀQtoy =\n', np.round(Qtoy.T @ Qtoy, 10))
print('→ diagonal = each column dotted with itself (squared length = 1), off-diagonal = dot products of different columns (perpendicular = 0): exactly the identity matrix')

# 2) reduced QR vs full QR: what really happens when the shapes don't match
A32 = np.array([[1., 2.], [3., 4.], [5., 6.]])
Q_reduced, R_reduced = np.linalg.qr(A32, mode='reduced')
Q_full, R_full = np.linalg.qr(A32, mode='complete')
print('\nreduced QR:  Q shape =', Q_reduced.shape, ' R shape =', R_reduced.shape)
print('complete QR: Q shape =', Q_full.shape, ' R shape =', R_full.shape)
print('Both modes satisfy Q@R = A:',
      np.allclose(Q_reduced @ R_reduced, A32), np.allclose(Q_full @ R_full, A32))
print('→ reduced keeps only the directions that A\'s two columns actually use; complete fills Q out to a full 3×3 rotation,')
print('  and the extra 3rd column has nothing to do with A\'s column space — it is only there to make Q a square matrix in the "standard" sense.')


## Code experiment: one matrix, three decompositions

Run the same matrix A through all three decompositions and compare what each one reports: the diagonals of L and U, the diagonal of R, and the singular values — Σ's diagonal values set how much stretching happens along each direction.

In [ ]:
import numpy as np

# Comparing three decompositions of the same matrix
A = np.array([[2.,1.,1.],[4.,3.,3.],[8.,7.,9.]])
# LU
# LU decomposition (Doolittle, no pivoting, pure numpy)
def _lu(A):
    n = len(A)
    L, U = np.eye(n), A.copy().astype(float)
    for j in range(n):
        for i in range(j+1, n):
            L[i,j] = U[i,j] / U[j,j]
            U[i] -= L[i,j] * U[j]
    return L, U
L, U = _lu(A)
print(f'LU: L diagonal={np.round(np.diag(L),4)}, U diagonal={np.round(np.diag(U),4)}')
# QR
Q, R = np.linalg.qr(A)
print(f'QR: R diagonal={np.round(np.diag(R),4)}')
# SVD
s = np.linalg.svd(A, compute_uv=False)
print(f'SVD: singular values={np.round(s,4)}')
print(f'rank={np.linalg.matrix_rank(A)}')


## 1. A = LU (Gaussian elimination in matrix form)

Lower triangular L × upper triangular U. This matches the elimination you learned in `L15`.

**Why it's done this way**: this section uses laviz pictures rather than bare formulas, because every factor has a matching geometric move, and a picture says it more directly than symbols. Take SVD: A=UΣVᵀ reads as three steps — Vᵀ rotates the input onto the matrix's principal directions, Σ stretches along those directions by the size of each singular value, and U rotates into the output space; laviz shows the three steps together, so the structure of the decomposition is readable at a glance. LU works the same way: the below-diagonal entries of L are the elimination multipliers (row shears), and U is the upper-triangular staircase — together the two steps record every row operation of Gaussian elimination.

In [ ]:
A = np.array([[2.,1,1],[4,3,3],[8,7,9]])
# Compute L, U with the _lu(A) algorithm (the same row operations as in L15)
L, U = _lu(A)
assert np.allclose(L @ U, A)
show_factorization(A, [L, U], ['L (lower triangular)', 'U (upper triangular)'],
                   modes=['#8AB17D', '#577590'], title='A = L U');

### Pause: why is L's diagonal always 1?

Recall what Gaussian elimination does: to turn the first number in row 2 into 0, you do "row 2 ← row 2 − multiplier × row 1"; to clear row 3, you then do "row 3 ← row 3 − multiplier × row 1 − multiplier × row 2"… The "multiplier" used in each operation is recorded in the matching position of L (for example `L[2,1]` records "how many times row 1 was subtracted from row 2").

So what do L's diagonal entries, like `L[1,1]` and `L[2,2]`, record? By the same logic they would record "how many times row i was subtracted from row i" — but that never happens: Gaussian elimination only uses rows **above**, already processed, to clear the leading numbers of the **current** row; it never uses a row to clear itself (that would turn the whole row into 0, which is pointless).

Another way to see it: expanding the matrix product $L U = A$, row $i$ of $A$ $= \sum_k L_{ik} \times (\text{row } k \text{ of } U)$. Since L is lower triangular, row $i$ has only its first $i$ entries non-zero, and if the $i$-th one (the one on the diagonal) were not 1, it would multiply "U's own row i" by some coefficient other than 1 — then U's row i would no longer be "the staircase row i left untouched once elimination reached this step", but some multiple of it, which breaks the fact that "U is the result of elimination".

So we set $L_{ii} = 1$: it means "row i contributes to itself with a coefficient of exactly 1, unchanged". That's no coincidence; it's dictated by the elimination process itself. Below we take the 3×3 elimination apart row by row and print it, so you can see the multiplier at each step and why the diagonal stays at 1 from start to finish.

In [ ]:
import numpy as np

# Extra experiment: trace a 3x3 elimination by hand to see why L's diagonal is always 1
A_demo = np.array([[2., 1, 1], [4, 3, 3], [8, 7, 9]])
n = len(A_demo)
L_demo, U_demo = np.eye(n), A_demo.copy().astype(float)
for j in range(n):
    print(f'--- clearing column {j} ---')
    for i in range(j + 1, n):
        multiplier = U_demo[i, j] / U_demo[j, j]
        L_demo[i, j] = multiplier
        U_demo[i] -= multiplier * U_demo[j]
        print(f'  row {i} -= {multiplier:.4f} × row {j}   →  L[{i},{j}] = {multiplier:.4f}')
    print('  current L diagonal:', np.diag(L_demo), '(1 throughout, because no row ever needs to "subtract a multiple of itself")')

print('\nFinal L =\n', np.round(L_demo, 4))
print('Final U =\n', np.round(U_demo, 4))


## 2. A = QR (Gram-Schmidt orthogonalization)

Orthogonal matrix Q × upper triangular R, with `QᵀQ = I`.

**Recap**: in the earlier section A was a 3×2 rectangular matrix, so Q was 3×2 as well (reduced QR). Here A is a 3×3 square matrix, so reduced QR and full QR coincide, Q is square 3×3, and `QᵀQ=I` really is "3 pairwise-perpendicular unit vectors".

In [ ]:
A = np.array([[1.,1,0],[1,0,1],[0,1,1]])
Q, R = np.linalg.qr(A)
assert np.allclose(Q @ R, A) and np.allclose(Q.T @ Q, np.eye(3))
show_factorization(A, [Q, R], ['Q (orthogonal)', 'R (upper triangular)'],
                   modes=['#2A9D8F', '#577590'], title='A = Q R');

## 3. S = QΛQᵀ (eigendecomposition of a symmetric matrix / the spectral theorem)

Symmetric matrix → orthogonal eigenvectors Q + diagonal eigenvalues Λ. This matches the diagonalization you learned in `L17`.

In [ ]:
S = np.array([[3.,3,3],[3,-1,1],[3,1,-1]])    # the same example as in L17
vals, Q = np.linalg.eigh(S)
Lam = np.diag(vals)
assert np.allclose(Q @ Lam @ Q.T, S)
show_factorization(S, [Q, Lam, Q.T], ['Q', 'Λ', 'Qᵀ'],
                   modes=['#2A9D8F', '#E9C46A', '#2A9D8F'], title='S = Q Λ Qᵀ');

### Pause: why are a symmetric matrix's eigenvectors automatically perpendicular to each other?

Picture a symmetric matrix as a "fair funhouse mirror": whichever direction you look in from, the way it "distorts" each direction follows one set of mutually balanced rules, and you never get the lopsided kind of warping that "sacrifices direction A's information to favor direction B". This "fairness" (mathematically, $S=S^\top$) forces its principal axes (its eigenvectors) to be perpendicular to each other — that's no coincidence, and it can be proved.

**Proof sketch (dot products only; high-school level is enough to follow)**:

Let $v_1, v_2$ be two eigenvectors of the symmetric matrix $S$ with two **different** eigenvalues $\lambda_1 \neq \lambda_2$, that is, $Sv_1 = \lambda_1 v_1$ and $Sv_2 = \lambda_2 v_2$.

Compute $v_1 \cdot (Sv_2)$: replacing $Sv_2$ with $\lambda_2 v_2$ gives $v_1\cdot(Sv_2) = \lambda_2 (v_1\cdot v_2)$.

Now compute it another way: because $S$ is symmetric ($S=S^\top$), there's a key little trick — you can move $S$ from one side of the dot product to the other without changing the result (the transpose has no effect on the dot product), so $v_1\cdot(Sv_2) = (Sv_1)\cdot v_2$. And $(Sv_1)\cdot v_2 = (\lambda_1 v_1)\cdot v_2 = \lambda_1(v_1\cdot v_2)$.

Both routes compute **the same number**, so:

$$\lambda_2 (v_1\cdot v_2) = \lambda_1 (v_1\cdot v_2) \quad\Longrightarrow\quad (\lambda_1-\lambda_2)(v_1\cdot v_2) = 0$$

Since $\lambda_1\neq\lambda_2$, the only option is $v_1\cdot v_2=0$ — the two eigenvectors must be perpendicular! That is a geometric result that follows directly from the algebraic condition "symmetric", $S=S^\top$. A general (non-symmetric) matrix doesn't have $S=S^\top$, so the equality in the second step fails, and its eigenvectors have no reason to be perpendicular.

**So how is the symmetric $S=Q\Lambda Q^\top$ related to the general $A=U\Sigma V^\top$?**

Go back to the "rotate–stretch–rotate" picture: in general, "turning the input into alignment" (the job of $V^\top$) and "finally turning it into place" (the job of $U$) can be two completely different rotations — because the input space and the output space may not even have the same dimension. But a symmetric matrix is special: it is always square, its input space and output space are **the same** space, and the "fairness" we just proved means "turning into alignment" and "turning into the final position" can be done with the very same set of axes, with no need for two different bases. That's why the formula for a symmetric matrix has only one Q (playing both "U on the left" and "V on the right").

Also, the eigenvalues in $\Lambda$ can be negative (meaning "flip along this direction, then stretch"), whereas the singular values in SVD's $\Sigma$ are required to be non-negative — because the sign of each direction has already been split off into the separate rotations U and V, and $\Sigma$ keeps only the pure "stretch factors". You could say: **SVD is the most general formula, designed for any matrix; $S=Q\Lambda Q^\top$ is the simplified version for the special case of a symmetric matrix** — symmetry saves you the trouble of "do I need two different bases?".

Below, use the S and Q from just now to check the key equality in the proof yourself.

In [ ]:
# Extra experiment: check for yourself the equality behind "symmetric ⟹ orthogonal eigenvectors"
v1, v2 = Q[:, 0], Q[:, 1]
lhs = v1 @ (S @ v2)   # v1 · (S v2)
rhs = (S @ v1) @ v2   # (S v1) · v2
print('v1·(Sv2) =', round(lhs, 6), '   (Sv1)·v2 =', round(rhs, 6), ' → the two are equal: this is S=Sᵀ at work')
print('v1·v2 =', round(v1 @ v2, 10), ' → since the two eigenvalues differ, only "perpendicular" can make the equality above hold')


## 4. A = UΣVᵀ (singular value decomposition, SVD — every matrix has one)

Left singular vectors U + singular values Σ + right singular vectors Vᵀ. The SVD from `L14`, drawn as a picture here.

### Pause: what exactly is a singular value? Why not just call it a "stretch factor"?

Imagine kneading a lump of modeling clay. The kneading can be split into three steps: ① first turn the clay to line it up with a few particular axes; ② stretch or squash it along each of those axes by some factor; ③ finally turn it again into the final position you want. That's exactly what SVD says — any matrix $A$ (that is, any linear transformation) can be split into three steps "rotate → stretch → rotate": $A = U\Sigma V^\top$.

A **singular value** is the number "how many times it was stretched" in step ②. So why not just call it a "stretch factor" instead of inventing a new name? Because it is a close relative of the "eigenvalue" but not quite the same thing: eigenvalues are only defined for **square** matrices, and they require "the direction is unchanged by the stretch" (a vector along an eigenvector still points the same way after the transformation); singular values are defined for matrices of **any shape** (even with different numbers of rows and columns), and "the direction before stretching" (a direction in the input space) and "the direction it lands in after stretching" (a direction in the output space) can be completely different — after all, input and output may not even be the same space. To tell these two different kinds of "factor" apart, they got separate names, "eigenvalue" and "singular value". How the two relate: for a **symmetric matrix**, the singular values are the absolute values of the eigenvalues; for a general matrix, the singular values are the square roots of the eigenvalues of $A^\top A$ (or $AA^\top$).

Before the 2×3 example below, look at a 2×2 example with no detours — this matrix is already diagonal and does nothing but "stretch", so the singular values coincide exactly with the "stretch factors", and you can check it at a glance.

In [ ]:
import numpy as np

# Extra experiment: the simplest 2x2 example — singular values are the stretch factors, plain to see
A2 = np.array([[3., 0.], [0., 1.]])   # stretches only, no rotation
U2, s2, Vt2 = np.linalg.svd(A2)
print('singular values =', s2)
print('Feed e1=(1,0) to A2:', A2 @ np.array([1., 0.]), ' → its length becomes', s2[0], '(stretched 3 times)')
print('Feed e2=(0,1) to A2:', A2 @ np.array([0., 1.]), ' → its length becomes', s2[1], '(stretched 1 time, i.e. unchanged)')

print('\nNow the shapes: when A is non-square, like 2×3, why do U/Σ/Vᵀ have these shapes?')
print('There are two "versions" here, the same story as reduced QR / full QR earlier:')
print('  · Full version (`np.linalg.svd(A)` default full_matrices=True):')
print('      Vᵀ is n×n and rotates the whole n-dimensional input space; Σ is m×n, the same shape as A (padded with 0s); U is m×m.')
print('  · Economy version (the notebook code below passes full_matrices=False explicitly):')
print('      keeps only the r=min(m,n) "actually useful" directions: Vᵀ becomes r×n, Σ a clean r×r square, U becomes m×r.')
print('Both versions give exactly the same A; the economy version just drops in advance the directions whose stretch factor is 0 anyway, so computing them is wasted —')
print('the natural result of the same idea as "drop the small singular values for a low-rank approximation" at the end of this lesson.')

A23 = np.array([[3., 1., 1.], [1., 3., 1.]])   # 2×3, the same A as in the next cell
U_full, s_full, Vt_full = np.linalg.svd(A23, full_matrices=True)
U_eco, s_eco, Vt_eco = np.linalg.svd(A23, full_matrices=False)
print('\nfull version    full_matrices=True :  U', U_full.shape, ' Σ diagonal', s_full.shape, ' Vᵀ', Vt_full.shape)
print('economy version full_matrices=False:  U', U_eco.shape, ' Σ diagonal', s_eco.shape, ' Vᵀ', Vt_eco.shape)
print('→ the next cell uses the economy version, so the Vᵀ you see is 2×3 and Σ(=diag(s)) is 2×2, not 3×3.')


In [ ]:
A = np.array([[3.,1,1],[1,3,1]])
U, s, Vt = np.linalg.svd(A, full_matrices=False)
Sig = np.diag(s)
assert np.allclose(U @ Sig @ Vt, A)
show_factorization(A, [U, Sig, Vt], ['U', 'Σ', 'Vᵀ'],
                   modes=['#2A9D8F', '#E9C46A', '#E76F51'], title='A = U Σ Vᵀ');

### Pause: take the 2×3 example above apart and compute it step by step

The `A` above has 2 rows and 3 columns, so it turns a 3-dimensional input vector into a 2-dimensional output vector. The code above uses the "economy" SVD (`full_matrices=False`), so here `Vt` has shape 2×3 and `Sig` is 2×2 — the `Vᵀ` step already combines "turning" with "keeping only the 2 useful directions", squeezing the 3-dimensional input straight down to 2 dimensions. Follow the code as it prints the shape at each step, and see exactly where this "combining" happens.

In [ ]:
# Extra experiment: track the shape at each step of the economy A = U Σ Vᵀ
x_test = np.array([1., 0., 0.])   # a vector in the 3-dimensional input space

print('Vt shape =', Vt.shape, ' Sig shape =', Sig.shape, ' U shape =', U.shape)

step1 = Vt @ x_test    # step 1: Vᵀ(2×3) projects the 3-D input straight to 2-D — "turning" and "keeping the useful directions" in one go
step2 = Sig @ step1    # step 2: Σ(2×2) stretches by the singular values within these 2 dimensions
step3 = U @ step2      # step 3: U(2×2) turns once more in the 2-D output space

print('\nOriginal input x (3-D):                          ', x_test)
print('Step 1 Vᵀx (now 2-D, already turned + compressed):', np.round(step1, 4))
print('Step 2 ΣVᵀx (still 2-D, stretched):               ', np.round(step2, 4))
print('Step 3 UΣVᵀx (2-D, final output):                 ', np.round(step3, 4))
print('A@x computed directly:                            ', np.round(A @ x_test, 4), ' ← should match step 3')


**🔗 Aurora**: SVD is used in recommendation (matrix factorization), embedding dimensionality reduction and LoRA; symmetric eigendecomposition = PCA. Next up, `L21`: aim these "matrix views" at Aurora's audio core.

### Pause: the code below checks U with `U.T@U=I` but Vt with `Vt@Vt.T=I` — isn't that inconsistent?

They say the same thing; only "whose rows, whose columns" differs. Recall what we said earlier: "a matrix's columns are pairwise orthogonal" $\iff$ the matrix's transpose times itself $=I$. That is, to check "are the columns of matrix M orthogonal", the formula is always $M^\top M = I$.

U itself is in "standard form" — the **columns** of U are the left singular vectors, so the check is written $U^\top U = I$, naturally.

But the variable `Vt` is already the transpose of $V$ (`Vt = V.T`)! So "the columns of V", seen through the variable `Vt`, become "the **rows** of `Vt`" (transposing turns columns into rows). If we want to check "the columns of V are orthogonal" but only have `Vt` in hand, how do we write it?

- If we first recover `V = Vt.T` and then check "the columns of V are orthogonal", it's written `V.T @ V = I`.
- But the shorter way is to work on `Vt` directly: "check that the rows of Vt are orthogonal", which is `Vt @ Vt.T = I` (a matrix times its own transpose checks the table of **row**-with-row dot products; the logic mirrors the earlier $M^\top M=I$ check of column-with-column exactly, with "column" swapped for "row").

So `U.T@U=I` and `Vt@Vt.T=I` look different, but they are "checking the **columns** of U" and "checking the **rows** of Vt" respectively — and the rows of Vt are exactly the columns of V. The two formulas arrive at the same place: both say "this orthonormal basis is pairwise perpendicular with length 1". Below, the code prints the rows of `Vt` next to the columns of `V`, so you can see it for yourself.

In [ ]:
# Extra experiment: why the "orthogonality checks" for U and Vt look different (reusing the A=UΣVᵀ example above)
V_full = Vt.T   # recover V itself
print('Is row 0 of Vt the same as column 0 of V:', np.allclose(Vt[0], V_full[:, 0]))
print('\nU checks "columns" orthogonal:       U.T @ U =\n', np.round(U.T @ U, 10))
print('\nVt checks "rows" orthogonal:         Vt @ Vt.T =\n', np.round(Vt @ Vt.T, 10))
print('\nEquivalently, V checks "columns":    V.T @ V =\n', np.round(V_full.T @ V_full, 10))
print('\n→ all three say the same thing: this orthonormal basis is pairwise perpendicular with length 1; only the "subject" switches between U/Vt/V.')


In [ ]:
import numpy as np

# Assertions on the mathematical properties of the big decompositions (standalone, no student implementation needed)
A = np.array([[3., 1., 1.], [1., 3., 1.]])   # 2×3 matrix

# SVD: A = U Σ Vᵀ, U and V both orthogonal, singular values non-negative and in descending order
U, s, Vt = np.linalg.svd(A, full_matrices=False)
assert np.allclose(U @ np.diag(s) @ Vt, A, atol=1e-12), "SVD reconstruction A=UΣVᵀ failed"
assert np.allclose(U.T @ U, np.eye(U.shape[1]), atol=1e-12), "U should satisfy UᵀU=I (orthonormal columns)"
assert np.allclose(Vt @ Vt.T, np.eye(Vt.shape[0]), atol=1e-12), "Vt should satisfy Vt·Vtᵀ=I (orthonormal rows of Vt = orthonormal columns of V)"
assert all(s >= 0), "singular values should be non-negative"
assert all(s[i] >= s[i+1] for i in range(len(s)-1)), "singular values should be in descending order"
print(f"SVD ✅  singular values={np.round(s, 4)}, descending, U/V have orthonormal columns")

# QR: Q has orthonormal columns, R is upper triangular (signs of the diagonal vary by implementation, but QR=A holds)
B = np.array([[1.,1.,0.],[1.,0.,1.],[0.,1.,1.]])
Q, R = np.linalg.qr(B)
assert np.allclose(Q @ R, B, atol=1e-12), "QR reconstruction A=QR failed"
assert np.allclose(Q.T @ Q, np.eye(Q.shape[1]), atol=1e-12), "Q should satisfy QᵀQ=I"
assert np.allclose(np.tril(R, -1), 0, atol=1e-12), "R should be upper triangular"
print(f"QR  ✅  R diagonal={np.round(np.diag(R), 4)}, QᵀQ=I, R upper triangular")

# Rank-deficient matrix: number of near-zero singular values = dimension of the null space
A_rank1 = np.outer([1., 2., 3.], [4., 5., 6.])
s_rank1 = np.linalg.svd(A_rank1, compute_uv=False)
zero_singular = np.sum(s_rank1 < 1e-10)
assert zero_singular == 2, f"a rank-1 3×3 matrix should have 2 zero singular values, got {zero_singular}"
print(f"Rank-deficient ✅  outer-product singular values={np.round(s_rank1, 6)}, 2 are ≈0, rank=1")
print("\nAll decomposition properties verified ✅")

### Pause: what is an outer product? Why does a rank-1 matrix have 2 zero singular values? And what does "rank deficient" mean?

**Definition of the outer product**: given a column vector $u$ ($m$ numbers) and another vector $v$ ($n$ numbers), the outer product $u\otimes v$ (in code, `np.outer(u, v)`) is defined as an $m\times n$ matrix whose $(i,j)$ entry is $u_i \times v_j$. In other words, row $i$ of the matrix $= u_i \times v$ (the whole vector $v$ times one coefficient).

**Why is an outer product always rank 1?** Because every row of the matrix is a multiple of the same vector $v$, all the rows are linearly dependent on each other — the single direction $v$ is enough to express every row. The matrix's "row space" is therefore only 1-dimensional, which is exactly the definition of "rank 1": the space spanned by the matrix's rows (or columns) has dimension 1.

**Why do 2 zero singular values show up in the SVD?** Recall what SVD does: "along a few mutually perpendicular directions, stretch by some factor each". If all the information of a $3\times 3$ matrix can be expressed completely "along the single direction $v$", then along the two other directions perpendicular to $v$ the matrix doesn't stretch at all — the stretch factor is 0. Singular values measure precisely "how much stretching happens along each orthogonal direction", so a direction with stretch factor 0 corresponds to a singular value of 0. So "the rank of a matrix = the number of non-zero singular values" is no coincidence; it follows from the definition of singular values itself.

**"Rank deficient"** is just the name for "the matrix's rank is smaller than the maximum its shape allows": a $3\times 3$ matrix can have rank at most 3 (all three directions independent, all stretched); here it has only rank 1, 2 short of "full rank", hence rank deficient.

Below, we reuse the `A_rank1` and `s_rank1` already computed in the previous code block and take a closer look.

In [ ]:
# Extra experiment: why is an outer product rank 1? Take the A_rank1 above apart
print('A_rank1 =\n', A_rank1)
print('row 0 / row 1 =', np.round(A_rank1[0] / A_rank1[1], 6),
      ' (should all equal u[0]/u[1] = 1/2 = 0.5, because both rows are multiples of the same vector [4,5,6])')
print('singular values =', np.round(s_rank1, 6), '→ only the first is non-zero; the "stretch factor" along the other two directions is 0')


### Pause: what is the "condition number"? Why does solving become unstable when it's large?

Imagine measuring something with a ruler whose markings have a tiny error (say every reading is off by about 0.1 mm). If the quantity you're measuring isn't sensitive to errors, the final result's error stays small; but if the quantity "amplifies" errors, a tiny reading error gets blown up into a large error in the result. The **condition number** is a number that measures "in the worst case, how many times the input error gets amplified".

**Mathematical definition**: for a matrix $A$, the condition number is $\kappa(A) = \sigma_{\max}/\sigma_{\min}$ (largest singular value divided by smallest singular value).

**Why can we read the condition number off the diagonal of QR's R below?** The diagonal entries of R also reflect how differently the matrix scales different directions (they aren't exactly the singular values, but they're likewise a measure of "how much each direction was stretched"). If the largest and smallest numbers on R's diagonal are wildly different (say 1000 and 1), the matrix stretches one direction 1000 times and barely stretches another (1 time) — that's "ill-conditioned".

**Why does this make "solving unstable"?** Solving $Ax=b$ amounts to "un-scaling" $b$ back along each direction. If some direction was stretched 1000 times, solving has to "shrink it by 1000" to undo it. Every floating-point operation a computer does carries a tiny rounding error (on the order of $10^{-16}$), and when that error gets amplified 1000 times by the undoing (or $10^{10}$ times for a worse-conditioned matrix), the computed solution can end up far from the true one. That's the essence of numerical instability: an error small enough to ignore gets amplified by the matrix's own structure into one you can't ignore.

Below we build a matrix with condition number 1000, add a tiny perturbation to $b$, and see for ourselves how much the solution $x$ gets amplified.

In [ ]:
import numpy as np

# Extra experiment: large condition number → a tiny error amplified many times (you need a perturbation in the "worst direction" to see the amplification)
A_bad = np.array([[1000., 0.], [0., 1.]])   # condition number = 1000
b = np.array([1000., 0.])                     # b lies mainly along the "strongly stretched" direction
x_exact = np.linalg.solve(A_bad, b)

# The perturbation goes along the "barely stretched" direction (the 2nd component), where errors get no "compression" at all
b_perturbed = b + np.array([0., 1e-3])
x_perturbed = np.linalg.solve(A_bad, b_perturbed)

print('exact solution x =', x_exact)
print('perturbed solution x =', x_perturbed)
rel_b = np.linalg.norm(b_perturbed - b) / np.linalg.norm(b)
rel_x = np.linalg.norm(x_perturbed - x_exact) / np.linalg.norm(x_exact)
print('relative perturbation of b:', rel_b)
print('relative change of x:', rel_x)
print('amplification =', rel_x / rel_b, '   κ(A) =', np.linalg.cond(A_bad), ' → same order of magnitude')
print('→ the key: the perturbation lands exactly on the direction the matrix "barely stretches", so the matrix structure does not "shrink" it,')
print('  and the tiny error passes into the solution untouched; relative to the already-small x, it looks "amplified" many times over.')


In [ ]:
import numpy as np

# Parameter experiment: matrices with different condition numbers, and the numerical stability of the decompositions
for cond_target in [1, 10, 1000]:
    s_vals = np.array([float(cond_target), 1.0])
    A = np.array([[s_vals[0],0.],[0.,s_vals[1]]])
    kappa = np.linalg.cond(A)
    _, r = np.linalg.qr(A)
    print(f'κ={kappa:.0f}  QR R diagonal={np.round(np.diag(r),4)}')
print('→ the larger the condition number, the wider the gap between R\'s diagonal entries, and the less stable solving the system becomes.')


## Parameter experiment: turn one knob at a time

In the SVD picture, change the matrix from full rank to rank 1 (e.g. `np.outer([1,2],[3,4])`) and observe that only the first singular value in Σ is non-zero, meaning only one direction's information is kept.

## 5. ✏️ Exercise: compute the numerical rank with SVD, `rank_from_svd(A)`

What `np.linalg.matrix_rank` really does: take the SVD and count how many singular values are > threshold.
Implement it yourself — calling `np.linalg.matrix_rank` is not allowed.

**Line of reasoning**:
1. `U, S, Vt = np.linalg.svd(A)`
2. Set the threshold `t = threshold * S[0]` (relative to the largest singular value)
3. Return the count of `S > t`

### Pause: why is the threshold "relative" (`threshold * S[0]`) rather than one fixed small number?

An analogy: to decide whether someone is "short", you wouldn't say "under 1 meter counts as short", because that standard means something completely different for a baby and for an adult — you'd say "a lot shorter than the average for their age", which is a relative standard.

Singular values work the same way: the sizes of different matrices' singular values can differ enormously — one matrix's largest singular value might be 1000, another's 0.001. If you used one fixed absolute threshold for all of them (say `1e-10`):

- When a matrix's singular values are all very small (say the largest is only `1e-5`), a `1e-10` threshold is relatively too strict: hardly any singular value will fall below it, so directions that should count as "close to 0, negligible" get treated as "effective" ones, and the rank comes out too high.
- Conversely, when a matrix's singular values are all very large (say the largest is `10000`), `1e-10` is too loose: any "fake singular value" produced by a little numerical error could be misjudged as an effective direction.

Using `threshold * S[0]` (`S[0]` is the largest singular value) "ties" the threshold to the matrix's own scale — however large or small the matrix's numbers are, the threshold scales along with them. So the standard for "how small counts as 0" is relative to this matrix itself, not one absolute number that applies everywhere. Below, two matrices of wildly different scale compare the verdicts of an absolute threshold and a relative threshold directly.

In [ ]:
import numpy as np

# Extra experiment: absolute threshold vs relative threshold
A_tiny = np.array([[1e-5, 0.], [0., 1e-6]])   # singular values all very small, but still full rank (2 independent directions)
s_tiny = np.linalg.svd(A_tiny, compute_uv=False)
print('singular values of A_tiny:', s_tiny)
print('number of "effective directions" by absolute threshold 1e-10:', np.sum(s_tiny > 1e-10), '(right by luck, only because the matrix values are not too small)')

A_big = np.array([[1e8, 0.], [0., 1e-3]])     # one direction stretched hugely, the other barely stretched
s_big = np.linalg.svd(A_big, compute_uv=False)
print('\nsingular values of A_big:', s_big)
print('number of "effective directions" by absolute threshold 1e-10:', np.sum(s_big > 1e-10),
      '(counts the almost useless direction too — 1e-3 next to 1e8 is already at noise level)')
print('number of "effective directions" by relative threshold (1e-10 * S[0]):', np.sum(s_big > 1e-10 * s_big[0]))


In [ ]:
import numpy as np

def rank_from_svd(A, threshold=1e-10):
    """Compute a matrix's numerical rank with SVD (without calling np.linalg.matrix_rank)"""
    A = np.asarray(A, float)
    # ✏️ TODO: svd → count the singular values > threshold * S[0]
    raise NotImplementedError('Please implement rank_from_svd')

# Check
try:
    # full-rank matrix
    A_full = np.array([[1.,2.],[3.,4.]])
    assert rank_from_svd(A_full) == 2, 'a full-rank 2×2 should return rank 2'

    # rank-deficient matrix (dependent rows)
    A_rank1 = np.outer([1., 2., 3.], [4., 5., 6.])  # outer product → rank 1
    assert rank_from_svd(A_rank1) == 1, f'a rank-1 matrix should return 1, got {rank_from_svd(A_rank1)}'

    # rectangular matrix
    A_rect = np.array([[1.,0.,0.],[0.,1.,0.]])       # 2×3, rank 2
    assert rank_from_svd(A_rect) == 2, 'a rectangular full-row-rank matrix should return 2'

    # compare with numpy
    rng = np.random.default_rng(42)
    A_rand = rng.standard_normal((5, 4))
    assert rank_from_svd(A_rand) == np.linalg.matrix_rank(A_rand), 'does not match numpy'

    print('✅ Passed: rank_from_svd is correct in all four cases: full rank, rank-deficient, rectangular, random')
except (NotImplementedError, TypeError):
    print('⬆️  Implement rank_from_svd first, then run this')


### Pause: what can you do once you have the rank? Try a "low-rank approximation / denoising" yourself

Picture an old photo that picked up some fine specks of noise over time and when it was scanned. If the photo's main content is simple (say a pattern made of a few lines), its "real information" can be described completely by a handful of "patterns"; the specks, meanwhile, spread almost evenly across all the small singular values (because random noise doesn't favor any particular direction). Dropping those small singular values amounts to filtering away "the minor, noise-like details" and keeping only "the few dominant patterns".

**The concrete steps**:
1. Take the SVD: `U, S, Vt = np.linalg.svd(A)`
2. Keep only the first `k` singular values and set the rest to 0: `S_truncated = S.copy(); S_truncated[k:] = 0`
3. Rebuild: the "compressed" matrix `= U @ np.diag(S_truncated) @ Vt`. In the matrix product, the columns of U and V that go with the zeroed singular values contribute nothing to the result (0 times anything is still 0), so you can multiply just the first k columns and get the same result with less computation.

Below we build a "true signal + noise" example and check whether the truncated version really is closer to the "true signal".

In [ ]:
import numpy as np

# Extra experiment: truncated SVD = low-rank approximation = removing the "noise-like" part
rng = np.random.default_rng(0)
true_signal = np.outer([1., 2., 3., 4.], [1., -1., 1.])   # the real information: rank 1
noise = 0.05 * rng.standard_normal(true_signal.shape)       # a little random noise
A_noisy = true_signal + noise

U_n, s_n, Vt_n = np.linalg.svd(A_noisy, full_matrices=False)
print('singular values =', np.round(s_n, 4), ' ← the 1st is far larger than the rest, which are all at noise level')

k = 1  # keep only the largest singular value (the true signal is rank 1 to begin with)
s_k = s_n.copy()
s_k[k:] = 0
A_denoised = U_n @ np.diag(s_k) @ Vt_n

print('\nerror vs true signal before denoising:', np.round(np.linalg.norm(A_noisy - true_signal), 4))
print('error vs true signal after denoising: ', np.round(np.linalg.norm(A_denoised - true_signal), 4))
print('→ after dropping the small singular values it is actually closer to the "true signal", because the noise hides mostly in those small singular values')


## Wrap-up

You can now get U, Σ and Vᵀ with `np.linalg.svd(A)`, and use the sequence of singular values to judge a matrix's effective rank and how much information each direction carries. In this lesson's `rank_from_svd` you wrote out by hand the criterion "count the singular values above the threshold" — it belongs with L18's rank criterion and L15's `classify_system` to the same question: "how much independent information can a matrix express?"

These decompositions map directly onto the low-rank approximation of Aurora's mel spectrogram matrix: keeping the first k singular values is enough to rebuild the main spectral structure. Next up, **L21** aims the matrix view at the audio core: the DFT matrix / mel matrix = multiplying by a filter bank.


### Pause: what does it really mean for the mel spectrogram matrix to be "low rank"? A toy version first

A mel filter bank is a row of "triangle"-shaped weighting templates; each template "gathers" the spectrum's energy within some frequency range into a single number. Neighboring triangles usually overlap (so the transition across frequency is smooth, with no abrupt edge effects).

This "heavily overlapping, mutually similar" structure means the templates aren't fully independent — each template can, to some extent, be "interpolated" from its neighbors. So the "true degrees of freedom" of the whole template matrix are far fewer than its number of rows (the number of templates). That's the intuition behind "low rank": the matrix has many rows (or columns), but they are highly correlated and can be rebuilt from far fewer "basic patterns" than there are rows — the same thing as the "truncated SVD low-rank approximation" earlier.

Below we build a simplified toy version (the real mel filter bank is covered in detail in `L21`) and see for ourselves whether, after an SVD, the singular values of "a pile of overlapping triangles" really do decay quickly toward 0.

In [ ]:
import numpy as np

# Extra experiment (preview of L21): why a "bank of overlapping triangular filters" is naturally low rank
def triangular_filter(center, width, n_points):
    x = np.arange(n_points)
    return np.maximum(0, 1 - np.abs(x - center) / width)

n_freq, n_filters = 40, 6
filterbank = np.array([
    triangular_filter(center=6 + i * 5, width=6, n_points=n_freq)
    for i in range(n_filters)
])
s_fb = np.linalg.svd(filterbank, compute_uv=False)
print('filter bank shape:', filterbank.shape)
print('singular values:', np.round(s_fb, 3))
print(f'→ the first few singular values are far larger than the later ones, so these {n_filters} "overlapping triangle" templates are not independent of each other;')
print('  a few "directions" are enough to roughly rebuild the whole filter bank — this is the "low rank" property L21 will use.')


### Pause: before the self-review, look at this "pass mark" reference

The self-review table below is only True/False, which can leave you unsure "how far you have to get to count as a pass". Here is a concrete test for each item:

- **Can you say what U, Σ and Vᵀ each mean?** Pass mark: given any matrix A, without looking at the notebook, you can explain "Vᵀ turns the input vector onto a few particular directions (with the economy SVD, it also keeps only the useful directions); Σ tells me how many times each direction is stretched; U turns the result into the final output directions". If some singular value is 0, you can say this means A does no stretching at all along that direction (information lost / not enough rank).
- **rank_from_svd implemented and passing its asserts?** Pass mark: run the exercise cell above and every `assert` passes; and you can explain "why the threshold is multiplied by S[0]" (hint: the earlier "absolute threshold vs relative threshold" extra experiment).
- **Understand that truncating singular values = low-rank approximation?** Pass mark: you can explain why "dropping the small singular values" is roughly "denoising" (hint: noise is usually spread evenly across all directions, including those where the true signal is weak; the true signal tends to concentrate in the directions of a few large singular values).
- **Read the L21 matrix-filtering preview in the wrap-up?** Pass mark: you can state the intuition behind "why a mel filter bank is naturally low rank" (hint: see the "overlapping triangles" extra experiment above — the templates overlap and can be interpolated from each other, so the degrees of freedom are far fewer than the number of templates).

If there's an item where you can't give an explanation at this level, go back and reread the matching "Pause" section — that's more useful than ticking True straight away.

In [ ]:
# ✏️ Self-review for this lesson
l20_review = {
    "svd_parts_understood":   None,  # can say what U, Σ, Vᵀ each mean? True/False
    "rank_from_svd_done":     None,  # rank_from_svd implemented and passes its asserts? True/False
    "low_rank_intuition":     None,  # understand truncated singular values = low-rank approximation? True/False
    "l21_bridge_read":        None,  # read the L21 matrix-filtering preview in the wrap-up? True/False
}

unfilled = [k for k, v in l20_review.items() if v is None]
assert not unfilled, f'Not filled in yet: {unfilled}'
weak = [k for k, v in l20_review.items() if v is False]
if weak:
    print(f'⚠️  Needs more work: {weak}')
else:
    print('✅ L20 complete! On to L21: matrices as filters')


---

→ **Next lesson** · [L21 · Matrices as filters](L21_aurora_matrices.ipynb)

> Next lesson covers **matrices as filters**: the DFT matrix and the mel matrix — audio processing is matrix multiplication.